In [ ]:
# Download Europarl v7 FR-EN directly (no Drive needed)
!wget -q http://www.statmt.org/europarl/v7/fr-en.tgz
!tar -xzf fr-en.tgz
print('Done downloading.')

In [ ]:
from collections import defaultdict

fr_path = 'europarl-v7.fr-en.fr'
en_path = 'europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
    fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
    en_lines = f.readlines()

TRAIN_SIZE = 50000

def tokenize(sentence):
    return sentence.lower().strip().split()

train_fr = [tokenize(s) for s in fr_lines[:TRAIN_SIZE]]
train_en = [tokenize(s) for s in en_lines[:TRAIN_SIZE]]

print(f'Train pairs: {len(train_fr):,}')
print(f'Sample: {train_fr[0]} -> {train_en[0]}')

In [ ]:
# Initialize translation table
fr_to_en = defaultdict(set)
for fr_sentence, en_sentence in zip(train_fr, train_en):
    for fr_word in fr_sentence:
        for en_word in en_sentence:
            fr_to_en[fr_word].add(en_word)

t = defaultdict(float)
for fr_word, en_words in fr_to_en.items():
    for en_word in en_words:
        t[(en_word, fr_word)] = 1.0 / len(en_words)

print(f'Translation table entries: {len(t):,}')

In [ ]:
# EM training — same 10 iterations as original
iterations = 10
for i in range(iterations):
    count = defaultdict(float)
    total = defaultdict(float)
    for en_sentence, fr_sentence in zip(train_en, train_fr):
        for en_word in en_sentence:
            sums = sum(t[(en_word, fr_word)] for fr_word in fr_sentence)
            if sums == 0:
                continue
            for fr_word in fr_sentence:
                delta = t[(en_word, fr_word)] / sums
                count[en_word, fr_word] += delta
                total[fr_word] += delta
    for en_word, fr_word in count:
        if total[fr_word] == 0:
            continue
        t[(en_word, fr_word)] = count[en_word, fr_word] / total[fr_word]
    print(f'Iteration {i+1}/{iterations}')

print('Training done.')

In [ ]:
# Build best-translation lookup and translate
best_translation = {}
for (en_word, fr_word), prob in t.items():
    if fr_word not in best_translation or prob > best_translation[fr_word][1]:
        best_translation[fr_word] = (en_word, prob)

def translate(fr_sentence):
    return ' '.join(
        best_translation[w][0] if w in best_translation else '<UNK>'
        for w in fr_sentence
    )

sentences = [
    "L'affaire NSA souligne l'absence totale de débat sur le renseignement",
    "Selon moi, il y a deux niveaux de réponse de la part du gouvernement français.",
]

for fr in sentences:
    tokens = tokenize(fr)
    print(f'FR:  {fr}')
    print(f'IBM: {translate(tokens)}')
    print()